## Resultados, documentação, governança e validação da Gold

In [0]:
%sql
SELECT
  'VoeBem Analytics' AS projeto,
  'Resultados da camada Gold' AS objetivo;

### 1. Tabelas Gold

In [0]:
%sql
SELECT
  table_name,
  table_type,
  comment
FROM voebem.information_schema.tables
WHERE table_schema = 'gold'
  AND table_name IN (
    'dim_aeroporto',
    'fato_voos',
    'obt_voos'
  )
ORDER BY table_name;

### 2. Quantidade de registros por tabela


In [0]:
%sql
SELECT
  'dim_aeroporto' AS tabela,
  COUNT(*) AS registros
FROM voebem.gold.dim_aeroporto

UNION ALL

SELECT
  'fato_voos' AS tabela,
  COUNT(*) AS registros
FROM voebem.gold.fato_voos

UNION ALL

SELECT
  'obt_voos' AS tabela,
  COUNT(*) AS registros
FROM voebem.gold.obt_voos

ORDER BY registros DESC;

### 3. Evolução da quantidade de registros


In [0]:
%sql
SELECT
  'Bronze VRA' AS camada,
  COUNT(*) AS registros
FROM voebem.bronze.vra

UNION ALL

SELECT
  'Silver VRA' AS camada,
  COUNT(*) AS registros
FROM voebem.silver.vra

UNION ALL

SELECT
  'Gold Fato Voos' AS camada,
  COUNT(*) AS registros
FROM voebem.gold.fato_voos

UNION ALL

SELECT
  'Gold OBT Voos' AS camada,
  COUNT(*) AS registros
FROM voebem.gold.obt_voos

ORDER BY
  CASE camada
    WHEN 'Bronze VRA' THEN 1
    WHEN 'Silver VRA' THEN 2
    WHEN 'Gold Fato Voos' THEN 3
    WHEN 'Gold OBT Voos' THEN 4
  END;

Databricks visualization. Run in Databricks to view.

### 4. Documentação das colunas Gold


In [0]:
%sql
SELECT
  table_name AS tabela,
  COUNT(*) AS total_colunas,
  COUNT(comment) AS colunas_comentadas,
  COUNT(*) - COUNT(comment) AS sem_comentario,
  ROUND(
    100.0 * COUNT(comment) / COUNT(*),
    2
  ) AS percentual_documentado
FROM voebem.information_schema.columns
WHERE table_schema = 'gold'
  AND table_name IN (
    'obt_voos',
    'fato_voos',
    'dim_aeroporto'
  )
GROUP BY table_name
ORDER BY tabela;

### 5. Quantidade de colunas por tabela


In [0]:
%sql
SELECT
  table_name AS tabela,
  COUNT(*) AS quantidade_colunas
FROM voebem.information_schema.columns
WHERE table_schema = 'gold'
  AND table_name IN (
    'obt_voos',
    'fato_voos',
    'dim_aeroporto'
  )
GROUP BY table_name
ORDER BY quantidade_colunas DESC;

### 6. Estrutura da OBT


In [0]:
%sql
SELECT
  ordinal_position,
  column_name,
  data_type,
  comment
FROM voebem.information_schema.columns
WHERE table_schema = 'gold'
  AND table_name = 'obt_voos'
ORDER BY ordinal_position;

### 7. Tags das tabelas Gold


In [0]:
%sql
SELECT
  table_name AS tabela,
  tag_name,
  tag_value
FROM voebem.information_schema.table_tags
WHERE schema_name = 'gold'
  AND table_name IN (
    'obt_voos',
    'fato_voos',
    'dim_aeroporto'
  )
ORDER BY
  table_name,
  tag_name;

### 8. Quantidade de tags por tabela


In [0]:
%sql
SELECT
  table_name AS tabela,
  COUNT(*) AS quantidade_tags
FROM voebem.information_schema.table_tags
WHERE schema_name = 'gold'
  AND table_name IN (
    'obt_voos',
    'fato_voos',
    'dim_aeroporto'
  )
GROUP BY table_name
ORDER BY tabela;

### 9. Minutos recuperados x atraso na chegada


In [0]:
%sql
SELECT
  CASE
    WHEN minutos_recuperados > 0
         AND atraso_chegada_min > 15
      THEN 'Recuperou tempo, mas chegou >15 min atrasado'

    WHEN minutos_recuperados > 0
         AND atraso_chegada_min <= 15
      THEN 'Recuperou tempo e chegou <=15 min atrasado'

    WHEN minutos_recuperados <= 0
      THEN 'Não recuperou tempo'

    ELSE 'Sem informação'
  END AS categoria,

  COUNT(*) AS quantidade_voos

FROM voebem.gold.obt_voos

GROUP BY
  CASE
    WHEN minutos_recuperados > 0
         AND atraso_chegada_min > 15
      THEN 'Recuperou tempo, mas chegou >15 min atrasado'

    WHEN minutos_recuperados > 0
         AND atraso_chegada_min <= 15
      THEN 'Recuperou tempo e chegou <=15 min atrasado'

    WHEN minutos_recuperados <= 0
      THEN 'Não recuperou tempo'

    ELSE 'Sem informação'
  END

ORDER BY quantidade_voos DESC;

### 10. Validação específica da descrição da métrica


In [0]:
%sql
SELECT
  COUNT(*) FILTER (
    WHERE minutos_recuperados > 0
  ) AS recuperaram_tempo,

  COUNT(*) FILTER (
    WHERE minutos_recuperados > 0
      AND atraso_chegada_min > 15
  ) AS recuperaram_mas_chegaram_atrasados,

  COUNT(*) FILTER (
    WHERE minutos_recuperados > 0
      AND atraso_chegada_min <= 15
  ) AS recuperaram_e_chegaram_com_ate_15_min,

  ROUND(
    100.0 *
    COUNT(*) FILTER (
      WHERE minutos_recuperados > 0
        AND atraso_chegada_min > 15
    )
    /
    NULLIF(
      COUNT(*) FILTER (
        WHERE minutos_recuperados > 0
      ),
      0
    ),
    2
  ) AS percentual_recuperaram_mas_atrasaram

FROM voebem.gold.obt_voos;

### 11. Validação de partida_pontual para voos cancelados


In [0]:
%sql
SELECT
  situacao_voo,
  partida_pontual,
  COUNT(*) AS quantidade_voos
FROM voebem.gold.obt_voos
WHERE situacao_voo IN ('CANCELADO', 'REALIZADO')
GROUP BY
  situacao_voo,
  partida_pontual
ORDER BY
  situacao_voo,
  partida_pontual;

Databricks visualization. Run in Databricks to view.

### 12. Validação de mes_referencia


In [0]:
%sql
SELECT
  CASE
    WHEN mes_referencia IS NULL THEN 'NULL'
    ELSE 'Preenchido'
  END AS status_mes_referencia,
  COUNT(*) AS quantidade_voos
FROM voebem.gold.obt_voos
GROUP BY
  CASE
    WHEN mes_referencia IS NULL THEN 'NULL'
    ELSE 'Preenchido'
  END
ORDER BY status_mes_referencia;

### 13. Atraso médio por companhia


In [0]:
%sql
SELECT
  icao_empresa,
  COUNT(*) AS quantidade_voos,
  ROUND(AVG(atraso_partida_min), 2) AS atraso_medio_partida,
  ROUND(AVG(atraso_chegada_min), 2) AS atraso_medio_chegada
FROM voebem.gold.obt_voos
WHERE situacao_voo = 'REALIZADO'
GROUP BY icao_empresa
HAVING COUNT(*) >= 100
ORDER BY atraso_medio_chegada DESC;

### 14. Aeroportos com maior quantidade de atrasos


In [0]:
%sql
SELECT
  icao_destino,
  COUNT(*) AS voos_atrasados
FROM voebem.gold.obt_voos
WHERE situacao_voo = 'REALIZADO'
  AND atraso_chegada_min > 15
GROUP BY icao_destino
ORDER BY voos_atrasados DESC
LIMIT 15;

Databricks visualization. Run in Databricks to view.

### 15. Rotas com maior quantidade de atrasos


In [0]:
%sql
SELECT
  CONCAT(icao_origem, ' → ', icao_destino) AS rota,
  COUNT(*) AS voos_atrasados
FROM voebem.gold.obt_voos
WHERE situacao_voo = 'REALIZADO'
  AND atraso_chegada_min > 15
GROUP BY
  icao_origem,
  icao_destino
ORDER BY voos_atrasados DESC
LIMIT 15;

### 16. Lineage relacionado à OBT


In [0]:
%sql
SELECT
  event_time,
  source_table_full_name,
  target_table_full_name,
  entity_type
FROM system.access.table_lineage
WHERE
  target_table_full_name = 'voebem.gold.obt_voos'
  OR source_table_full_name = 'voebem.gold.obt_voos'
ORDER BY event_time DESC;

### 17. Resumo final do projeto


In [0]:
%sql
SELECT
  'Bronze VRA' AS etapa,
  COUNT(*) AS registros
FROM voebem.bronze.vra

UNION ALL

SELECT
  'Silver VRA',
  COUNT(*)
FROM voebem.silver.vra

UNION ALL

SELECT
  'Gold Fato Voos',
  COUNT(*)
FROM voebem.gold.fato_voos

UNION ALL

SELECT
  'Gold OBT Voos',
  COUNT(*)
FROM voebem.gold.obt_voos;

Databricks visualization. Run in Databricks to view.

### Gráfico 1 — Registros por tabela

In [0]:
%sql
SELECT 'dim_aeroporto' AS tabela, COUNT(*) AS registros
FROM voebem.gold.dim_aeroporto

UNION ALL

SELECT 'fato_voos', COUNT(*)
FROM voebem.gold.fato_voos

UNION ALL

SELECT 'obt_voos', COUNT(*)
FROM voebem.gold.obt_voos;

Databricks visualization. Run in Databricks to view.

### Gráfico 2 — Bronze → Silver → Gold

In [0]:
%sql
SELECT
  'Bronze VRA' AS camada,
  COUNT(*) AS registros
FROM voebem.bronze.vra

UNION ALL

SELECT
  'Silver VRA',
  COUNT(*)
FROM voebem.silver.vra

UNION ALL

SELECT
  'Gold Fato Voos',
  COUNT(*)
FROM voebem.gold.fato_voos

UNION ALL

SELECT
  'Gold OBT Voos',
  COUNT(*)
FROM voebem.gold.obt_voos;

Databricks visualization. Run in Databricks to view.

### Gráfico 3 — Documentação

In [0]:
%sql
SELECT
  table_name AS tabela,
  COUNT(*) AS total_colunas,
  COUNT(comment) AS colunas_comentadas,
  ROUND(100.0 * COUNT(comment) / COUNT(*), 2) AS percentual_documentado
FROM voebem.information_schema.columns
WHERE table_schema = 'gold'
  AND table_name IN (
    'obt_voos',
    'fato_voos',
    'dim_aeroporto'
  )
GROUP BY table_name
ORDER BY tabela;

Databricks visualization. Run in Databricks to view.

### Gráfico 4 — minutos_recuperados

In [0]:
%sql
SELECT
  CASE
    WHEN minutos_recuperados > 0
         AND atraso_chegada_min > 15
      THEN 'Recuperou tempo, mas chegou >15 min atrasado'

    WHEN minutos_recuperados > 0
         AND atraso_chegada_min <= 15
      THEN 'Recuperou tempo e chegou <=15 min atrasado'

    WHEN minutos_recuperados <= 0
      THEN 'Não recuperou tempo'

    ELSE 'Sem informação'
  END AS categoria,
  COUNT(*) AS quantidade
FROM voebem.gold.obt_voos
GROUP BY
  CASE
    WHEN minutos_recuperados > 0
         AND atraso_chegada_min > 15
      THEN 'Recuperou tempo, mas chegou >15 min atrasado'

    WHEN minutos_recuperados > 0
         AND atraso_chegada_min <= 15
      THEN 'Recuperou tempo e chegou <=15 min atrasado'

    WHEN minutos_recuperados <= 0
      THEN 'Não recuperou tempo'

    ELSE 'Sem informação'
  END
ORDER BY quantidade DESC;

Databricks visualization. Run in Databricks to view.